# Method C: Gap Filler (revs 383-401)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kduvekot/ubl-gc/blob/claude/run-local-validation-lafAX/notebooks/method-c-gap-filler.ipynb)

## Purpose

The main slow-validation run downloaded 1986/2005 library revisions
successfully. Exactly **19 revisions** are missing from Drive:
**rev 383 through rev 401**.

These revisions were originally downloaded in the fast bulk run but
the files were never saved to Drive (the bulk run collapsed at ~383
and the slow-validation had them in `done_revs` so it skipped them).

## Strategy

- Download revs **370-420** (51 revisions)
- Revs 370-382: warm-up / verify (files already on Drive)
- Revs 383-401: the actual gap (19 missing files)
- Revs 402-420: verify after the gap
- **No batch cooldowns** (we learned cooldowns cause collapse!)
- **Always save** historical downloads to Drive
- **Separate** results/checkpoint files (won't touch the main run's data)

## Expected runtime

51 revisions × 10s delay = ~8.5 minutes

In [ ]:
# === Step 0: Auth + Mount ===
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

import google.auth
from google.auth.transport.requests import Request as AuthRequest
creds, project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/drive']
)
creds.refresh(AuthRequest())
TOKEN = creds.token
print(f'Token: {TOKEN[:15]}...{TOKEN[-4:]}')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/ubl-gc-revisions')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive: {DRIVE_DIR}')

## Step 1: Configuration

Targeted at the gap: revs 370-420, library sheet only.

In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │  CONFIGURATION — gap filler for revs 383-401                   │
# └─────────────────────────────────────────────────────────────────┘

# Delay between API calls (seconds).
REQUEST_DELAY = 10.0

# Stop after this many consecutive identical content hashes.
CANARY_THRESHOLD = 10

# No batch cooldowns! (cooldowns cause collapse)
BATCH_SIZE = 9999
COOLDOWN_MINUTES = 0

# Which sheet to download
TEST_SHEET = 'ubl25_library'

# The gap range: warm up from 370, fill 383-401, verify through 420
REV_START = 370
REV_END = 420
GAP_START = 383
GAP_END = 401

# Build test plan: revisions 370-420
TEST_PLAN = [(rev, 'gap-fill') for rev in range(REV_START, REV_END + 1)]

# Use separate results/checkpoint files (don't touch main run's data)
RUN_TAG = 'gap-383-401'

print(f'Gap filler: {TEST_SHEET}')
print(f'  Range: rev {REV_START} -> {REV_END} ({len(TEST_PLAN)} revisions)')
print(f'  Gap:   rev {GAP_START} -> {GAP_END} ({GAP_END - GAP_START + 1} missing files)')
print(f'  Delay: {REQUEST_DELAY}s between requests')
print(f'  Canary: stop after {CANARY_THRESHOLD} identical hashes in a row')
print(f'  Batch cooldowns: DISABLED')
est_minutes = len(TEST_PLAN) * REQUEST_DELAY / 60
print(f'  Estimated runtime: ~{est_minutes:.1f} min')

In [ ]:
# === Step 2: Helpers ===
import json, time, hashlib, gzip, zipfile, io, re
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from urllib.error import HTTPError

SHEETS = {
    'ubl25_library':   '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY',
    'ubl25_documents': '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg',
}

# Known "current content" hashes — if we see these, the API collapsed.
CURRENT_HASHES = {
    'ubl25_library': {
        '716eacf3ebe60d7622e4c9c439e3807b7cf035cd8581ad62c495b8d154064010',
        '7bd3c0e40778115f96c0e5515c093e23e15d1c7d3bbe8b34f29c80f5b97de823',
        '8cbf8ad66e4319af28d48df2e658122d943a402e145d528bdd2f6f25f7f9a6fe',
    },
    'ubl25_documents': {
        'd1301dedea016c962d0dbdb8f287bdd0f60c6911086e5f29f0bca389afa08ac3',
        '5fd9129713bef9336a68ea19821ab406e0e1d6317b1d4a2d7a5ca2aba8a7f8e7',
    },
}


def now_iso():
    return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S.%fZ')


def export_revision_ods(sheet_id, rev_num, retry_wait=9.0):
    """Method C: download a specific revision as ODS.
    Returns (ods_bytes, http_status, error_msg, hit_429).
    On 429: waits retry_wait seconds then retries (up to 2 retries)."""
    url = (f'https://docs.google.com/spreadsheets/export'
           f'?id={sheet_id}&revision={rev_num}&exportFormat=ods')
    headers = {'Authorization': f'Bearer {TOKEN}'}
    hit_429 = False
    for attempt in range(3):
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=120) as resp:
                data = resp.read()
                if len(data) > 500:
                    return data, 200, None, hit_429
                return None, 200, f'too small ({len(data)} bytes)', hit_429
        except HTTPError as e:
            if e.code == 429 and attempt < 2:
                hit_429 = True
                print(f'429({retry_wait:.0f}s)...', end='', flush=True)
                time.sleep(retry_wait)
                continue
            if e.code in (500, 502, 503) and attempt < 2:
                print(f'retry({e.code}, {retry_wait:.0f}s)...', end='', flush=True)
                time.sleep(retry_wait)
                continue
            return None, e.code, str(e.code), hit_429
        except Exception as exc:
            if attempt < 2:
                time.sleep(retry_wait)
                continue
            return None, 0, str(exc), hit_429
    return None, 0, 'max retries', hit_429


def ods_content_hash(ods_bytes):
    """Extract content.xml from ODS and SHA256 it."""
    try:
        with zipfile.ZipFile(io.BytesIO(ods_bytes)) as zf:
            return hashlib.sha256(zf.read('content.xml')).hexdigest()
    except Exception:
        return None


print('Helpers ready')

## Step 3: Pre-flight check

Verify Drive state before starting: which files exist, which are missing.

In [ ]:
# === Step 3: Pre-flight — check Drive state ===
current_hashes = CURRENT_HASHES[TEST_SHEET]
ods_dir = DRIVE_DIR / TEST_SHEET
ods_dir.mkdir(exist_ok=True)


def read_drive_ods_hash(rev_num):
    """Read an existing .ods.gz from Drive, decompress, extract
    content.xml, return its SHA256. Returns None if file missing."""
    gz_path = ods_dir / f'rev-{rev_num}.ods.gz'
    if not gz_path.exists():
        return None
    try:
        ods_bytes = gzip.decompress(gz_path.read_bytes())
        return ods_content_hash(ods_bytes)
    except Exception as e:
        print(f'  [warn] rev-{rev_num} decompress error: {e}')
        return None


# Check which files exist on Drive
print(f'Pre-flight check: revs {REV_START}-{REV_END}')
print(f'Drive dir: {ods_dir}')
print()
existing = 0
missing = 0
missing_revs = []
for rev in range(REV_START, REV_END + 1):
    gz_path = ods_dir / f'rev-{rev}.ods.gz'
    in_gap = GAP_START <= rev <= GAP_END
    if gz_path.exists():
        existing += 1
        tag = '  (expected: verify)'
        if in_gap:
            tag = '  (UNEXPECTED: gap rev has file!)'
        print(f'  rev-{rev:>4}: EXISTS{tag}')
    else:
        missing += 1
        missing_revs.append(rev)
        tag = '  (expected: will download)' if in_gap else '  (UNEXPECTED: should exist!)'
        print(f'  rev-{rev:>4}: MISSING{tag}')

print(f'\nSummary: {existing} existing, {missing} missing')
print(f'Missing revs: {missing_revs}')

gap_missing = [r for r in missing_revs if GAP_START <= r <= GAP_END]
print(f'Gap revs missing: {len(gap_missing)}/{GAP_END - GAP_START + 1}')
if len(gap_missing) == GAP_END - GAP_START + 1:
    print('All gap revisions need downloading — ready to proceed!')
elif len(gap_missing) == 0:
    print('All gap revisions already exist — nothing to do!')
else:
    print(f'Partial gap: {gap_missing}')

## Step 4: Download the gap

Downloads revs 370-420 at 10s pace. Key differences from the main run:

- **No batch cooldowns** (cooldowns cause collapse!)
- **Always saves** historical downloads to Drive
- **Separate** results/checkpoint files
- **No checkpoint resume** (fresh start, only 51 revisions)

In [ ]:
sheet_id = SHEETS[TEST_SHEET]
ods_dir.mkdir(exist_ok=True)

# ── Separate checkpoint + results files (don't touch main run) ──
checkpoint_path = DRIVE_DIR / f'checkpoint-{TEST_SHEET}-{RUN_TAG}.json'
results_path = DRIVE_DIR / f'slow-validation-{TEST_SHEET}-{RUN_TAG}.json'

# ── Adaptive backoff state ──
MIN_DELAY = 10.0
RETRY_GAP = 4.0
current_delay = REQUEST_DELAY
ok_streak = 0
speedup_threshold = 12
total_429s = 0
rate_changes = []
resume_after_rev = None

# ── Load checkpoint if resuming ──
if checkpoint_path.exists():
    cp = json.loads(checkpoint_path.read_text())
    resume_after_rev = cp.get('last_rev')
    current_delay = max(cp.get('delay', current_delay), MIN_DELAY)
    speedup_threshold = cp.get('speedup_threshold', speedup_threshold)
    ok_streak = cp.get('ok_streak', 0)
    total_429s = cp.get('total_429s', 0)
    print(f'Checkpoint loaded: resume after rev {resume_after_rev}, '
          f'delay={current_delay:.1f}s')
else:
    print(f'No checkpoint — starting fresh')


def save_checkpoint(rev_num):
    checkpoint_path.write_text(json.dumps({
        'sheet': TEST_SHEET,
        'run_tag': RUN_TAG,
        'last_rev': rev_num,
        'delay': round(current_delay, 1),
        'speedup_threshold': speedup_threshold,
        'ok_streak': ok_streak,
        'total_429s': total_429s,
        'timestamp': now_iso(),
    }, indent=2))


# ── Load or init results ──
if results_path.exists():
    results = json.loads(results_path.read_text())
    done_revs = {r['rev'] for r in results.get('tests', [])}
    print(f'Results file: {len(done_revs)} revisions already logged')
else:
    results = {
        'sheet_key': TEST_SHEET,
        'sheet_id': sheet_id,
        'run_tag': RUN_TAG,
        'config': {
            'request_delay': REQUEST_DELAY,
            'canary_threshold': CANARY_THRESHOLD,
            'rev_start': REV_START,
            'rev_end': REV_END,
            'gap_start': GAP_START,
            'gap_end': GAP_END,
        },
        'started': now_iso(),
        'tests': [],
        'summary': {},
    }
    done_revs = set()


def save_results():
    results['last_updated'] = now_iso()
    tests = results['tests']
    results['summary'] = {
        'total_tested': len(tests),
        'total_ok': sum(1 for t in tests if t['status'] == 'ok'),
        'total_error': sum(1 for t in tests if t['status'] == 'error'),
        'historical': sum(1 for t in tests if t.get('is_historical')),
        'current': sum(1 for t in tests if t.get('is_current')),
        'saved_to_drive': sum(1 for t in tests if t.get('saved_to_drive')),
        'gap_filled': sum(1 for t in tests
                         if t.get('is_historical') and t.get('in_gap')),
        'canary_triggered': results.get('canary_triggered', False),
    }
    results_path.write_text(json.dumps(results, indent=2))


# ── Skip logic ──
skip_up_to = resume_after_rev is not None
revs_skipped = 0

# ── Main download loop ──
print(f'\n{"="*70}')
print(f'Gap Filler: {TEST_SHEET} (revs {REV_START}-{REV_END})')
print(f'  Gap: revs {GAP_START}-{GAP_END} ({GAP_END - GAP_START + 1} missing)')
print(f'  delay={current_delay:.1f}s, canary={CANARY_THRESHOLD}')
print(f'  Batch cooldowns: DISABLED')
print(f'  Checkpoint: {checkpoint_path.name}')
print(f'  Results:    {results_path.name}')
print(f'{"="*70}\n')

consecutive_same = 0
last_hash = None
canary_triggered = False

for i, (rev_num, group) in enumerate(TEST_PLAN):
    # ── Skip: checkpoint resume ──
    if skip_up_to:
        if rev_num == resume_after_rev:
            skip_up_to = False
            revs_skipped += 1
            continue
        else:
            revs_skipped += 1
            continue

    # ── Skip: already done ──
    if rev_num in done_revs:
        continue

    in_gap = GAP_START <= rev_num <= GAP_END
    phase = 'GAP' if in_gap else ('warm-up' if rev_num < GAP_START else 'verify')

    # ── Download ──
    t0 = time.time()
    retry_wait = current_delay + RETRY_GAP
    print(f'  [{i+1}/{len(TEST_PLAN)}] rev-{rev_num:>4} ({phase:>7}): ',
          end='', flush=True)

    ods_data, http_status, error, hit_429 = export_revision_ods(
        sheet_id, rev_num, retry_wait=retry_wait)
    dl_elapsed = time.time() - t0

    # ── Adaptive backoff on 429 ──
    if hit_429:
        total_429s += 1
        ok_streak = 0
        old_delay = current_delay
        current_delay += 1.0
        speedup_threshold += 1

    if not ods_data:
        print(f'ERROR (HTTP {http_status}: {error}) [{dl_elapsed:.1f}s]')
        results['tests'].append({
            'rev': rev_num,
            'group': group,
            'phase': phase,
            'in_gap': in_gap,
            'status': 'error',
            'http_status': http_status,
            'error': error,
            'hit_429': hit_429,
            'timestamp': now_iso(),
            'elapsed': round(dl_elapsed, 2),
        })
        consecutive_same = 0
        last_hash = None
        save_results()
        save_checkpoint(rev_num)
        time.sleep(current_delay)
        continue

    # ── Adaptive: count successes ──
    if not hit_429:
        ok_streak += 1
        if ok_streak >= speedup_threshold:
            old_delay = current_delay
            current_delay = max(current_delay - 0.5, MIN_DELAY)
            ok_streak = 0
            speedup_threshold = max(speedup_threshold - 1, 6)

    # ── Hash ──
    content_hash = ods_content_hash(ods_data)
    ods_size = len(ods_data)
    is_current = content_hash in current_hashes
    is_historical = not is_current and content_hash is not None

    # ── Compare with existing Drive file ──
    drive_hash = read_drive_ods_hash(rev_num)
    file_existed = drive_hash is not None
    matches_drive = (drive_hash == content_hash) if file_existed else None

    # ── Canary ──
    if content_hash == last_hash:
        consecutive_same += 1
    else:
        consecutive_same = 1
        last_hash = content_hash

    # ── Build entry ──
    entry = {
        'rev': rev_num,
        'group': group,
        'phase': phase,
        'in_gap': in_gap,
        'status': 'ok',
        'content_hash': content_hash,
        'ods_size': ods_size,
        'is_current': is_current,
        'is_historical': is_historical,
        'file_existed': file_existed,
        'matches_drive': matches_drive,
        'consecutive_same': consecutive_same,
        'hit_429': hit_429,
        'delay_at_time': current_delay,
        'timestamp': now_iso(),
        'elapsed': round(dl_elapsed, 2),
        'saved_to_drive': False,
    }

    # ── Print status ──
    parts = [f'{ods_size:,}b']
    if is_current:
        parts.append('CURRENT')
    else:
        parts.append(f'HISTORICAL ({content_hash[:12]}...)')

    if file_existed:
        if matches_drive:
            parts.append('matches Drive')
        else:
            parts.append('DIFFERS from Drive')
    else:
        parts.append('no file on Drive')

    parts.append(f'[{dl_elapsed:.1f}s]')
    if consecutive_same > 1:
        parts.append(f'(same x{consecutive_same})')
    print(' '.join(parts))

    # ── Save to Drive: ALWAYS save historical downloads ──
    # Unlike the main run, we save regardless of what was there before.
    # For warm-up/verify revs this overwrites with (likely identical) data.
    # For gap revs this creates the missing files.
    if is_historical:
        gz_path = ods_dir / f'rev-{rev_num}.ods.gz'
        gz_data = gzip.compress(ods_data, compresslevel=6)
        gz_path.write_bytes(gz_data)
        entry['saved_to_drive'] = True
        action = 'FILLED GAP' if in_gap else ('verified' if file_existed else 'saved')
        print(f'         >>> {action}: {gz_path.name} '
              f'({len(gz_data):,} gz bytes)')

    results['tests'].append(entry)
    done_revs.add(rev_num)
    save_results()
    save_checkpoint(rev_num)

    # ── Canary ──
    if consecutive_same >= CANARY_THRESHOLD:
        print(f'\n  *** CANARY: {consecutive_same} identical hashes in a row ***')
        print(f'  *** Hash: {content_hash[:32]}...')
        print(f'  *** Collapsed to current content. Stopping.')
        results['canary_triggered'] = True
        results['canary_at_rev'] = rev_num
        save_results()
        canary_triggered = True
        break

    # ── Wait ──
    remaining_delay = max(0, current_delay - (time.time() - t0))
    if remaining_delay > 0:
        time.sleep(remaining_delay)

# ── Final save ──
results['completed'] = now_iso()
results['adaptive_backoff'] = {
    'final_delay': current_delay,
    'total_429s': total_429s,
    'rate_changes': rate_changes,
}
save_results()

if revs_skipped:
    print(f'\n  Skipped {revs_skipped} revisions (checkpoint)')
if not canary_triggered:
    print(f'\n  All {len(TEST_PLAN)} revisions completed!')
print(f'  Total 429s: {total_429s}')

## Step 5: Verify the gap is filled

In [ ]:
# === Step 5: Post-run verification ===
results = json.loads(results_path.read_text())
tests = results['tests']

print(f'{"="*70}')
print(f'GAP FILLER RESULTS: {TEST_SHEET}')
print(f'{"="*70}')
print(f'  Started:   {results.get("started", "?")}')
print(f'  Completed: {results.get("completed", "?")}')
print()

s = results.get('summary', {})
print(f'  Total tested:     {s.get("total_tested", 0)}')
print(f'  Successful (OK):  {s.get("total_ok", 0)}')
print(f'  Errors:           {s.get("total_error", 0)}')
print(f'  Historical:       {s.get("historical", 0)}')
print(f'  Current (broken): {s.get("current", 0)}')
print(f'  Saved to Drive:   {s.get("saved_to_drive", 0)}')
print(f'  Gap filled:       {s.get("gap_filled", 0)}/{GAP_END - GAP_START + 1}')
print(f'  Canary triggered: {s.get("canary_triggered", False)}')

# Detailed results
print(f'\n{"Rev":>5}  {"Phase":>7}  {"Size":>8}  {"Content":>10}  '
      f'{"Drive":>15}  {"Hash (first 16)"}')
print('-' * 80)
for t in tests:
    if t['status'] == 'error':
        print(f'{t["rev"]:>5}  {t["phase"]:>7}  {"ERROR":>8}  '
              f'HTTP {t.get("http_status", "?")}')
        continue
    content = 'CURRENT' if t.get('is_current') else 'historical'
    drive_status = ''
    if t.get('file_existed'):
        drive_status = 'match' if t.get('matches_drive') else 'DIFFERS'
    else:
        drive_status = 'NEW (gap!)' if t.get('in_gap') else 'no file'
    saved = ' SAVED' if t.get('saved_to_drive') else ''
    h = t.get('content_hash', '?')[:16]
    print(f'{t["rev"]:>5}  {t["phase"]:>7}  {t.get("ods_size",0):>8,}  '
          f'{content:>10}  {drive_status:>15}{saved}  {h}...')

# Final Drive verification
print(f'\n{"="*70}')
print(f'DRIVE VERIFICATION: checking files exist for gap revs')
print(f'{"="*70}')
all_filled = True
for rev in range(GAP_START, GAP_END + 1):
    gz_path = ods_dir / f'rev-{rev}.ods.gz'
    if gz_path.exists():
        size = gz_path.stat().st_size
        print(f'  rev-{rev}: OK ({size:,} gz bytes)')
    else:
        print(f'  rev-{rev}: MISSING!')
        all_filled = False

print()
if all_filled:
    print('  >>> ALL GAP REVISIONS FILLED! Library is now complete (2005/2005).')
else:
    print('  >>> Some gap revisions still missing. Check errors above.')

---

## For Claude: How to check progress

This notebook uses separate checkpoint/results files:
- `checkpoint-ubl25_library-gap-383-401.json`
- `slow-validation-ubl25_library-gap-383-401.json`

Use the same Drive folder listing approach, then download by file ID.
The run is short (~8.5 min) so it may already be done by the time you check.